# apertus-eval-prep — vLLM on Colab

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_vllm.ipynb)

Runtime → Change runtime type → **T4 GPU**.

This notebook scores the frozen slice with vLLM using **already-rendered** completion prompts (no second chat template). Download the JSON at the end and commit it to `results/vllm_tokenizer.json`.

In [ ]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e .

In [ ]:
# Colab T4: CUDA 12 host + modern vLLM (needs libcudart.so.13 from pip).
import os, sys, subprocess, glob
from pathlib import Path

def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', *a])

_pip('uninstall', '-y', 'vllm')
_pip('install', '-q', '--upgrade', 'nvidia-cuda-runtime==13.0.88', 'nvidia-cublas==13.0.2', 'nvidia-cuda-nvrtc==13.0.88')
lib_dirs=[]
for pat in ('/usr/local/lib/python*/dist-packages/nvidia/**/lib*.so*',):
    for p in glob.glob(pat, recursive=True):
        lib_dirs.append(str(Path(p).parent))
os.environ['LD_LIBRARY_PATH']=':'.join(dict.fromkeys(lib_dirs + [os.environ.get('LD_LIBRARY_PATH','')]))
_pip('install', '-q', 'vllm')
import torch
assert torch.cuda.is_available(), 'Set runtime to GPU (T4) and rerun.'
from vllm import LLM, SamplingParams  # smoke
import vllm
print(torch.cuda.get_device_name(0), 'vllm', getattr(vllm, '__version__', '?'))


In [ ]:
!python -m apertus_eval_prep dump-prompts --config configs/vllm.yaml --out results/prompts_vllm.txt --n 2
!python -m apertus_eval_prep eval --config configs/vllm.yaml --out results/vllm_tokenizer.json

In [ ]:
# Optional: also run HF generate on the same GPU for a same-hardware backend delta.
!python -m apertus_eval_prep eval --config configs/default.yaml --backend hf --out results/hf_tokenizer_colab.json
!python -m apertus_eval_prep compare results/hf_tokenizer_colab.json results/vllm_tokenizer.json --out results/compare_backend.md
print(open("results/compare_backend.md").read())

In [ ]:
import os
from google.colab import files
for name in [
    "results/vllm_tokenizer.json",
    "results/hf_tokenizer_colab.json",
    "results/compare_backend.md",
]:
    if os.path.exists(name):
        files.download(name)